In [10]:
# IMPORTS
# FIRST, pip install -r requirements.txt

import json
import joblib
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

W_CONTENT:   float = 0.80
W_COLLAB:    float = 0.20
FINAL_TOP_K: int   = 150

In [3]:
# PATHS
ROOT = Path(".")
DATA_RAW = ROOT / "data" / "raw"
DATA_PROC = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"
RESULTS_DIR = ROOT / "results" / "playlists"

for d in (DATA_PROC, MODELS_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

USER_FILES = {
    "andy": DATA_RAW / "enrichment-success-andy.json",
    "dishita": DATA_RAW / "enrichment-success-dish.json",
    "riya": DATA_RAW / "enrichment-success-riya.json",
    "priyanka": DATA_RAW / "enrichment-success-priyanka.json",
}

USERS = list(USER_FILES.keys())

# TUNEABLE PARAMETERS FOR ENTIRE PIPELINE
AUDIO_FEATURES = [
    "danceability", "energy", "valence", "tempo", "loudness",
    "acousticness", "instrumentalness", "speechiness", "liveness",
    "key", "mode", "time_signature",
]
MOOD_FEATURES = ["valence", "energy", "danceability", "tempo"]

# DATA LOADING & PREPROCESSING | TRACK TABLES
MIN_MEANINGFUL_MS = 30_000
TFIDF_MIN_DF = 2
TFIDF_MAX_FEATURES = 500

# USER PROFILES
MIN_SKIP_RATE_DISLIKE = 0.7
MAX_MS_DISLIKE = 15_000
N_MOOD_CLUSTERS = 4

# CANDIDATE GENERATION
TOP_N_CONTENT = 300
TOP_N_TAG = 200
TOP_N_NEIGHBOR = 150
FINAL_CANDIDATE_K = 1000
TOP_K_TAGS = 10
MIN_NEIGHBOR_MS = 30_000
MAX_NEIGHBOR_SKIP = 0.3
NEIGHBOR_PLAY_THRESHOLD = 1 
PRE_SCORE_CONTENT_W = 0.50
PRE_SCORE_TAG_W = 0.20
PRE_SCORE_GROUP_W = 0.15
PRE_SCORE_NEIGHBOR_W = 0.15

In [4]:
# DATA LOADING AND PREPROCESSING

"""
Returns the best available tag list for a track.
 - Tries all_tags first.
 - If empty or missing, falls back to sc_genres.
 - If still empty, falls back to lastfm_tags.
 - Returns a list of lowercase, stripped strings.
 - If none are available, returns an empty list.
"""
def resolve_tags_columns(df: pd.DataFrame) -> pd.Series:
    tags = df["all_tags"]

    mask = tags.apply(lambda t: not isinstance(t, list) or len(t) == 0)
    tags = tags.where(~mask, df["sc_genres"])

    mask = tags.apply(lambda t: not isinstance(t, list) or len(t) == 0)
    tags = tags.where(~mask, df["lastfm_tags"])

    tags = tags.apply(
        lambda t: [x.strip().lower() for x in t] if isinstance(t, list) else []
    )

    return tags

"""
Loads a single user's raw (enriched) JSON listening history and returns it as a cleaned df.
 - Filters out non-track plays (episodes, audiobooks)
 - Renames audio feature columns to simplify access/keep consistency
 - Converts timestamps to pandas datetime
 - Ensures numeric values for ms_played
 - Marks skipped tracks as boolean
 - Adds 'tags' column
 - Adds 'meaningful' column
"""
def load_user_events(path: Path, user: str) -> pd.DataFrame:
    with open(path, encoding="utf-8") as f:
        raw = json.load(f)

    df = pd.json_normalize(raw)
    df['user'] = user

    df.rename(columns={f"audio_features.{f}": f for f in AUDIO_FEATURES}, inplace=True)
    mask = df["spotify_track_uri"].notna() & df["episode_name"].isna()
    df = df.loc[mask]

    df["ts"] = pd.to_datetime(df["ts"], utc=True)
    df["ms_played"] = pd.to_numeric(df["ms_played"], errors="coerce").fillna(0)
    df["skipped"] = df["skipped"].fillna(False).astype(bool)
    df["tags"] = resolve_tags_columns(df)

    # Meaningful play: not skipped AND listened to >= 30 seconds
    df["meaningful"] = (~df["skipped"]) & (df["ms_played"] >= MIN_MEANINGFUL_MS)
    return df

"""
Loads and combines listening history for all users into a single df.
Each row represents one play event, tagged with the user it belongs to.
"""
def load_all_events(user_files: dict) -> pd.DataFrame:
    frames = []

    for user, path in user_files.items():
        df = load_user_events(path, user)
        frames.append(df)

    return pd.concat(frames, ignore_index=True)

events = load_all_events(USER_FILES)

# Save raw events for reference
events.to_csv(DATA_PROC / "events.csv", index=False)

print(f"Total play events: {len(events):,}")
print(f"Unique tracks: {events['spotify_track_uri'].nunique():,}")
print(f"Users: {events['user'].unique()}")

Total play events: 29,887
Unique tracks: 7,068
Users: ['andy' 'dishita' 'riya' 'priyanka']


In [5]:
# TRACK TABLE (ITEM VECTORS)

# COLUMNS TO INCLUDE IN THE TRACK TABLE
cols = [
    "spotify_track_uri",
    "master_metadata_track_name",
    "master_metadata_album_artist_name",
    "master_metadata_album_album_name",
    "tags",
    *AUDIO_FEATURES,
]

# BUILD TRACK TABLE: ONE ROW PER UNQIUE TRACK URI
track_table = (
    events[cols]
    .drop_duplicates("spotify_track_uri")
    .rename(columns={
        "master_metadata_track_name": "track_name",
        "master_metadata_album_artist_name": "artist_name",
        "master_metadata_album_album_name": "album_name",
    })
    .reset_index(drop=True)
)

# ASSIGN A UNIQUE INTEGER ID TO EACH TRACK
track_table["track_id"] = np.arange(len(track_table))

# LOOKUP DICTIONARIES (FOR FASTER MAPPING)
uri_to_id = track_table.set_index("spotify_track_uri")["track_id"].to_dict()
id_to_uri = track_table.set_index("track_id")["spotify_track_uri"].to_dict()

# NORMALIZED AUDIO FEATURE MATRIX
audio_matrix = track_table[AUDIO_FEATURES].fillna(0).to_numpy(dtype=float)
audio_scaler = MinMaxScaler()
audio_matrix_norm = audio_scaler.fit_transform(audio_matrix)

# TAG TF-IDF MATRIX
tag_docs = track_table["tags"].str.join(" ").fillna("")
tfidf = TfidfVectorizer(min_df=TFIDF_MIN_DF, max_features=TFIDF_MAX_FEATURES)
tag_matrix_sparse = tfidf.fit_transform(tag_docs)
tag_matrix = tag_matrix_sparse.toarray() # dense ver.
tag_vocab = tfidf.get_feature_names_out()

# CONVERT ARTIST NAME TO ARTIST ID
track_table["artist_id"] = pd.Categorical(track_table["artist_name"]).codes

# SAVE TRACK TABLE & FITTED MODELS
track_table.to_csv(DATA_PROC / "track_features.csv", index=False)
joblib.dump(audio_scaler, MODELS_DIR / "audio_scaler.joblib")
joblib.dump(tfidf, MODELS_DIR / "tfidf.joblib")

print(f"Track table saved: {len(track_table):,} tracks, tag vocab = {len(tag_vocab)}")

Track table saved: 7,068 tracks, tag vocab = 330


In [17]:
# USER PROFILES

# MAP TRACK URIS TO IDS
# Each track URI gets a unique integer ID for indexing into matrices.
events["track_id"] = events["spotify_track_uri"].map(uri_to_id)

# PER-(USER, TRACK) PLAY-STATS
"""
Aggregates raw play events to compute per-user, per-track statistics:
- n_plays: total plays of the track
- avg_ms: average milliseconds listened
- skip_count: number of skipped plays
- meaningful_pct: fraction of plays considered meaningful (>=30s, not skipped)
"""
play_stats = (
    events.groupby(["user", "track_id"])
    .agg(
        n_plays=("ms_played", "count"),
        avg_ms=("ms_played", "mean"),
        skip_count=("skipped", "sum"),
        meaningful_pct=("meaningful", "mean"),
    )
    .reset_index()
)

# Fraction of plays skipped for each user-track
play_stats["skip_rate"] = play_stats["skip_count"] / play_stats["n_plays"]

# PLAY WEIGHT (VECTORIZED)
"""
Assigns a weight to a (user, track) play record showing how much the user likes the track.
 - Tracks that are frequently skipped early get a weight of 0.
 - Otherwise, weight increases with the percentage of meaningful listens and 
    the log of total play count (to dampen the reduced impact of excessive replays).
"""
# log1p dampens obsessive replays — 10 plays shouldn't outweigh 2 plays by 5x
play_stats["weight"] = play_stats["meaningful_pct"] * np.log1p(play_stats["n_plays"])

# Zero out tracks the user consistently bails on early
dislike_mask = (
    (play_stats["skip_rate"] > MIN_SKIP_RATE_DISLIKE)
    & (play_stats["avg_ms"] < MAX_MS_DISLIKE)
)

play_stats.loc[dislike_mask, "weight"] = 0.0
play_stats.to_csv(DATA_PROC / "play_stats.csv", index=False)

# Map users to row indices for the sparse matrix
user_to_idx = {u: i for i, u in enumerate(USERS)}
play_stats["user_idx"] = play_stats["user"].map(user_to_idx)

# Build a sparse user × track weight matrix
# Rows = users, Columns = tracks, Values = play weight
user_track_matrix = csr_matrix(
    (play_stats["weight"].values, (play_stats["user_idx"].values, play_stats["track_id"].values)),
    shape=(len(USERS), audio_matrix_norm.shape[0])
)

# Precompute sum of weights per user to normalize later
weight_sum = np.array(user_track_matrix.sum(axis=1)).flatten()

# LONG-TERM AUDIO PROFILES
"""
Builds a single vector representing each user's overall audio taste.
- Weighted average of all tracks the user listened to.
- Tracks with higher engagement contribute more to the profile.
"""
# Matrix multiply gives weighted sum; divide by weight_sum to get weighted average
_ltp_matrix = user_track_matrix @ audio_matrix_norm
_ltp_matrix = _ltp_matrix / (weight_sum[:, None] + 1e-9) # +1e-9 avoids div-by-zero
long_term_profiles = {u: _ltp_matrix[user_to_idx[u]] for u in USERS}

# TAG DISTRIBUTIONS PER USER
"""
Builds weighted tag profiles for all users in one pass.
- Captures which genres and descriptors are characteristic of each user's taste.
- Weighted by play engagement.
"""
_utp_matrix = user_track_matrix @ tag_matrix
_utp_matrix = _utp_matrix / (weight_sum[:, None] + 1e-9)
user_tag_profiles = {u: _utp_matrix[user_to_idx[u]] for u in USERS}

# MOOD CLUSTERS PER USER
"""
Identifies mood-based clusters in a user's listening history.
- Uses a subset of audio features (MOOD_FEATURES).
- Weighted KMeans clustering per user.
- Returns cluster centroids, labels, track indices, and weighted tag distributions per cluster.
"""
# Extract mood features and fill NaNs with 0 before scaling
mood_feat_idx = [AUDIO_FEATURES.index(f) for f in MOOD_FEATURES]
mood_matrix = np.nan_to_num(audio_matrix[:, mood_feat_idx], nan=0.0)

# Scale to [0,1]
mood_scaler = MinMaxScaler()
mood_matrix_norm = mood_scaler.fit_transform(mood_matrix)

def build_mood_clusters(user: str):
    u_idx = user_to_idx[user]
    # .indices/.data give the non-zero columns and values from the sparse row
    idx = user_track_matrix[u_idx].indices
    w = user_track_matrix[u_idx].data

    # Handle users with no meaningful plays
    if len(idx) == 0:
        return np.empty((0, len(MOOD_FEATURES))), np.array([]), np.array([]), {}

    X = mood_matrix_norm[idx]
    k = min(N_MOOD_CLUSTERS, len(idx))
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X, sample_weight=w)

    tag_dist = {}
    for c in range(k):
        mask = labels == c
        c_idx, c_w = idx[mask], w[mask]
        if len(c_idx) == 0:
            tag_dist[c] = np.zeros(tag_matrix.shape[1])
        else:
            ws = np.array((tag_matrix[c_idx] * c_w[:, None]).sum(axis=0)).flatten()
            tag_dist[c] = ws / (c_w.sum() + 1e-9)
    return km.cluster_centers_, labels, idx, tag_dist

print("Building mood clusters...")
mood_clusters = {u: build_mood_clusters(u) for u in USERS}

# Save fitted mood scaler for later use
joblib.dump(mood_scaler, MODELS_DIR / "mood_scaler.joblib")

# SAVE LONG-TERM AUDIO PROFILES
# Each row = user
# Each column = audio feature
pd.DataFrame([long_term_profiles[u] for u in USERS], columns=AUDIO_FEATURES) \
    .assign(user=USERS) \
    .to_csv(DATA_PROC / "user_profiles.csv", index=False)

# SAVE TAG PROFILES
# Each row = user
# Each column = tag
pd.DataFrame([user_tag_profiles[u] for u in USERS], columns=tag_vocab) \
    .assign(user=USERS) \
    .to_csv(DATA_PROC / "user_tag_profiles.csv", index=False)

# SAVE MOOD CLUSTERS PER USER
# One CSV per user
# Rows = cluster centroids
for u in USERS:
    centroids, *_ = mood_clusters[u]
    pd.DataFrame(centroids, columns=MOOD_FEATURES) \
        .rename_axis("cluster") \
        .reset_index() \
        .assign(user=u) \
        .to_csv(DATA_PROC / f"mood_clusters_{u}.csv", index=False)
    
# USER-USER SIMILARITY
"""
Computes pairwise user similarity using the tag profiles.
Uses cosine similarity between all users.
"""
user_sim_df = pd.DataFrame(
    cosine_similarity(np.stack([user_tag_profiles[u] for u in USERS])),
    index=USERS, columns=USERS
)

print("\nUser profiles saved -> data/processed/")
print("mood_scaler.joblib -> models/")
print("\n", user_sim_df.round(3))


Building mood clusters...

User profiles saved -> data/processed/
mood_scaler.joblib -> models/

            andy  dishita   riya  priyanka
andy      1.000    0.835  0.534     0.906
dishita   0.835    1.000  0.628     0.818
riya      0.534    0.628  1.000     0.741
priyanka  0.906    0.818  0.741     1.000


In [7]:
# CANDIDATE GENERATION

"""
Finds tracks most similar to the user's taste using audio feature similarity. 
Queries against both the user's long-term profile and each of their mood centroids, returning 
the union of top matches. 
Tracks the user consistently skips early are excluded.
"""
def content_knn_candidates(user: str) -> set:
    # Tracks the user frequently skips early. Treated as a dislike
    bad = set(
        play_stats[
            (play_stats["user"] == user) & (play_stats["skip_rate"] > MIN_SKIP_RATE_DISLIKE)
        ]["track_id"]
    )

    base = long_term_profiles[user].copy()
    queries = [base]
    centroids, *_ = mood_clusters[user]

    # Build one query vector per mood centroid by splicing centroid values 
    # into the user's long-term profile
    for c in centroids:
        v = base.copy()
        
        for i, fi in enumerate(mood_feat_idx):
            v[fi] = c[i]
            
        queries.append(v)

    candidates = set()
    for qv in queries:
        sims = cosine_similarity(qv.reshape(1, -1), audio_matrix_norm)[0]
        top_idx = np.argsort(sims)[::-1][:TOP_N_CONTENT]
        candidates.update(int(i) for i in top_idx if i not in bad)

    return candidates

"""
Finds tracks that match the user's top genre/mood tags.
Also finds tracks that have been played by at least one person in the group. 
Expands the candidate pool with genre-relevant tracks validated by how much a track has been 
played by other users.
"""
def tag_expansion_candidates(user: str) -> set:
    # Column indices of the user's top-k tags in the TF-IDF vocab
    top_tag_idx = np.argsort(user_tag_profiles[user])[::-1][:TOP_K_TAGS]
    # Sum TF-IDF scores across top tags to rank every track by genre relevance
    tag_scores = np.array(tag_matrix[:, top_tag_idx].sum(axis=1)).flatten()

    group_relevant = set(
        play_stats.groupby("track_id")["user"]
        .nunique()
        .loc[lambda x: x >= 1]
        .index
    )

    candidates = set()
    for tid in np.argsort(tag_scores)[::-1]:
        if int(tid) in group_relevant:
            candidates.add(int(tid))
        if len(candidates) >= TOP_N_TAG:
            break

    return candidates

"""
Finds tracks that similar users (neighbors) listened to frequently, but the target user has not yet 
heard. 
Neighbors are weighted by their tag-based similarity to the target user.
(Tracks from more similar users in the group contribute more). 
(THE Collaborative filtering).
"""
def neighbor_expansion_candidates(user: str) -> set:
    neighbors = [u for u in USERS if u != user]
    sim_scores = {n: float(user_sim_df.loc[user, n]) for n in neighbors}

    # Tracks the user has already heard enough to be considered "known"
    u_played = set(play_stats[
        (play_stats["user"] == user) & (play_stats["n_plays"] > NEIGHBOR_PLAY_THRESHOLD)
    ]["track_id"])

    nbr_score: dict[int, float] = {}
    for nbr in neighbors:
        good = play_stats[
            (play_stats["user"] == nbr)
            & (play_stats["avg_ms"] >= MIN_NEIGHBOR_MS)
            & (play_stats["skip_rate"] <= MAX_NEIGHBOR_SKIP)
        ]

        for _, row in good.iterrows():
            tid = int(row["track_id"])
            if tid not in u_played:
                # Weight neighbor's signal by how similar they are to the target user
                nbr_score[tid] = (nbr_score.get(tid, 0) + sim_scores[nbr] * row["meaningful_pct"])

    ranked = sorted(nbr_score, key=nbr_score.get, reverse=True)
    return set(ranked[:TOP_N_NEIGHBOR])

"""
Merges all three candidate sources (content k-NN, tag expansion, neighbor expansion).
Computes a pre-score for each track combining:
 - Content similarity
 - Tag similarity
 - Group popularity
  - Neighbor signal
Returns the top FINAL_CANDIDATE_K tracks sorted by pre-score.
"""
def build_candidates(user: str) -> pd.DataFrame:
    content_cands = content_knn_candidates(user)
    tag_cands = tag_expansion_candidates(user)
    neighbor_cands = neighbor_expansion_candidates(user)
    all_cands = content_cands | tag_cands | neighbor_cands

    # Score every candidate against the full track matrix in one pass
    content_sims = cosine_similarity(long_term_profiles[user].reshape(1, -1),
                                     audio_matrix_norm)[0]

    tag_sims = cosine_similarity(user_tag_profiles[user].reshape(1, -1),
                                 tag_matrix)[0]
    
    # Fraction of the group (0–1) who have played this track
    group_pop = (
        play_stats.groupby("track_id")["user"]
        .nunique()
        .div(len(USERS))
        .to_dict()
    )

    # Re-compute neighbor signal across all tracks (not just neighbor_cands)
    # so every candidate gets a meaningful neighbor score, not just 0 or signal
    neighbors = [u for u in USERS if u != user]
    sim_scores = {n: float(user_sim_df.loc[user, n]) for n in neighbors}
    nbr_signal: dict[int, float] = {}

    for nbr in neighbors:
        good = play_stats[(play_stats["user"] == nbr) & (play_stats["avg_ms"] >= MIN_NEIGHBOR_MS)]
        for _, row in good.iterrows():
            tid = int(row["track_id"])
            nbr_signal[tid] = (nbr_signal.get(tid, 0) + sim_scores[nbr] * row["meaningful_pct"])

    # Normalize neighbor signal to [0, 1] so it's on the same scale as other scores
    if nbr_signal:
        max_nbr = max(nbr_signal.values())
        nbr_signal = {k: v / max_nbr for k, v in nbr_signal.items()}

    rows = []
    for tid in all_cands:
        cs = float(content_sims[tid])
        ts = float(tag_sims[tid])
        gp = float(group_pop.get(tid, 0))
        ns = float(nbr_signal.get(tid, 0))
        rows.append({
            "track_id": tid,
            "pre_score": (
                PRE_SCORE_CONTENT_W * cs
                + PRE_SCORE_TAG_W * ts
                + PRE_SCORE_GROUP_W * gp
                + PRE_SCORE_NEIGHBOR_W * ns
            ),
            "content_sim": cs,
            "tag_sim": ts,
            "group_pop": gp,
            "neighbor_score": ns,
            "from_content": tid in content_cands,
            "from_tag": tid in tag_cands,
            "from_neighbor": tid in neighbor_cands,
        })

    df = (
        pd.DataFrame(rows)
        .sort_values("pre_score", ascending=False)
        .head(FINAL_CANDIDATE_K)
        .reset_index(drop=True)
        .merge(
            track_table[["track_id", "track_name", "artist_name"]],
            on="track_id",
            how="left",
        )
    )

    return df

candidate_pools = {}
for user in USERS:
    candidate_pools[user] = build_candidates(user)
    candidate_pools[user].to_csv(DATA_PROC / f"candidates_{user}.csv", index=False)
    print(f"{user:10s}: {len(candidate_pools[user]):,} candidates")

print("\nCandidate generation complete.")

andy      : 1,000 candidates
dishita   : 1,000 candidates
riya      : 1,000 candidates
priyanka  : 1,000 candidates

Candidate generation complete.


In [21]:
#SCORING

# Helper function to help scale values to [0, 1] for fair combination into final score (that way one
# feature with a wider range doesn't dominate the final score)
def _minmax(arr: np.ndarray) -> np.ndarray:
    mn, mx = arr.min(), arr.max()
    return (arr - mn) / (mx - mn + 1e-9)

"""
This function determines the active session mood by looking at the user's most recently played tracks and seeing which
mood cluster they belong to.
"""
def _get_session_centroid(user, last_n=10):
    centroids, labels, idx, tag_dist = mood_clusters[user]

    if len(centroids) == 0:
        return None, None

    # get the user's most recently played track ids
    recent_track_ids = (
        events[events["user"] == user]
        .sort_values("ts", ascending=False)
        .drop_duplicates("track_id")
        .head(last_n)["track_id"]
        .values
    )

    # find which of those tracks exist in the user's mood cluster history
    recent_in_history = [t for t in recent_track_ids if t in idx]

    if not recent_in_history:
        # no overlap — fall back to first centroid
        return centroids[0], tag_dist[0]

    # for each recent track that's in the history, look up which cluster it belongs to
    idx_list    = list(idx)
    cluster_ids = [labels[idx_list.index(t)] for t in recent_in_history]

    # whichever cluster appears most among recent tracks is the active session mood
    active_cluster = max(set(cluster_ids), key=cluster_ids.count)
    return centroids[active_cluster], tag_dist[active_cluster]

"""
Extracts features for a given user and candidate tracks, including:
- Content similarity to the user's long-term profile
- Tag similarity to the user's tag profile
- Mood similarity to the active session mood centroid
- Cluster tag similarity to the active session mood's tag distribution
- Tag Jaccard similarity between the track's top tags and the user's top tags
- Group popularity (fraction of users who have played the track)
- Group average listen duration and skip rate
- Neighbor signal (weighted engagement from similar users)

Returns a DataFrame with one row per candidate track and all the extracted features.
"""

def _content_features(user, track_ids, session_mood_centroid, session_tag_dist, candidates_df):
    pre = candidates_df.set_index("track_id")

    if session_mood_centroid is not None:
        mood_query = np.zeros((1, audio_matrix_norm.shape[1]))
        for i, fi in enumerate(mood_feat_idx):
            mood_query[0, fi] = session_mood_centroid[i]
        all_mood_sims = cosine_similarity(mood_query, audio_matrix_norm)[0]
    else:
        all_mood_sims = np.zeros(audio_matrix_norm.shape[0])

    # cluster tag similarity — how well does this track match the active mood's tag profile
    # different from tag_cosine which uses the user's overall long-term tag profile
    if session_tag_dist is not None:
        all_cluster_tag_sims = cosine_similarity(session_tag_dist.reshape(1, -1), tag_matrix)[0]
    else:
        all_cluster_tag_sims = np.zeros(tag_matrix.shape[0])

    top_user_tag_idx_ranked = np.argsort(user_tag_profiles[user])[::-1][:3]
    top_user_tag_idx        = set(top_user_tag_idx_ranked)

    rows = []
    for tid in track_ids:
        track_tag_vec     = np.asarray(tag_matrix[tid]).flatten()
        top_track_tag_idx = set(np.argsort(track_tag_vec)[::-1][:5])
        intersection      = top_track_tag_idx & top_user_tag_idx
        union             = top_track_tag_idx | top_user_tag_idx

        rows.append({
            "track_id":        tid,
            "long_term_sim":   float(pre.loc[tid, "content_sim"]),        # from candidates_df
            "tag_cosine":      float(pre.loc[tid, "tag_sim"]),             # from candidates_df
            "mood_sim":        float(all_mood_sims[tid]),                  # new
            "cluster_tag_sim": float(all_cluster_tag_sims[tid]),           # new
            "tag_jaccard":     float(len(intersection) / len(union)) if union else 0.0,  # new
        })

    return pd.DataFrame(rows)

"""
Extracts collaborative features for candidate tracks, including:
- Group listener count (fraction of users who have played the track)
- Group average listen duration and skip rate
- Neighbor signal (weighted engagement from similar users)

Returns a DataFrame with one row per candidate track and all the extracted collaborative features.
"""
def _collab_features(user, track_ids, candidates_df):
    pre       = candidates_df.set_index("track_id")
    neighbors = [u for u in USERS if u != user]

    grp = (
        play_stats.groupby("track_id")
        .agg(
            group_avg_ms   =("avg_ms",    "mean"),
            group_skip_rate=("skip_rate", "mean"),
        )
        .to_dict("index")
    )

    nbr_weighted_ms = {}
    nbr_weight_sum  = {}
    for nbr in neighbors:
        w = float(user_sim_df.loc[user, nbr])
        for _, row in play_stats[play_stats["user"] == nbr].iterrows():
            tid = int(row["track_id"])
            nbr_weighted_ms[tid] = nbr_weighted_ms.get(tid, 0.0) + w * row["avg_ms"]
            nbr_weight_sum[tid]  = nbr_weight_sum.get(tid, 0.0)  + w

    rows = []
    for tid in track_ids:
        g  = grp.get(tid, {})
        ws = nbr_weight_sum.get(tid, 0.0)
        rows.append({
            "track_id":             tid,
            "group_listener_count": float(pre.loc[tid, "group_pop"]) * len(USERS),  # from candidates_df
            "group_avg_ms":         float(g.get("group_avg_ms",    0)),              # new
            "group_skip_rate":      float(g.get("group_skip_rate", 1)),              # new
            "weighted_nbr_avg_ms":  float(nbr_weighted_ms.get(tid, 0.0) / ws) if ws > 0 else 0.0,  # new
        })

    return pd.DataFrame(rows)

"""
Builds training labels for candidate tracks based on the user's play history.
- Liked (1): avg_ms > liked_ms threshold AND skip_rate < 20%
- Disliked (0): skip_rate > skip_thresh
- Neutral/Unknown (-1): everything else (including tracks the user hasn't played)

Returns a numpy array of labels corresponding to the input track_ids, which can be used for training a model
to learn feature weights.
"""

def _session_features(user, track_ids, last_n=10):
    recent_track_ids = (
        events[events["user"] == user]
        .sort_values("ts", ascending=False)
        .drop_duplicates("track_id")
        .head(last_n)["track_id"]
    )

    recent_stats = play_stats[
        (play_stats["user"] == user) &
        (play_stats["track_id"].isin(recent_track_ids))
    ]

    recent_skip_pct = float(recent_stats["skip_rate"].mean()) if len(recent_stats) else 0.5

    recent_meta        = track_table[track_table["track_id"].isin(recent_track_ids)]
    recent_avg_valence = float(recent_meta["valence"].mean()) if len(recent_meta) else 0.5
    recent_avg_energy  = float(recent_meta["energy"].mean())  if len(recent_meta) else 0.5

    rows = []
    for tid in track_ids:
        t            = track_table[track_table["track_id"] == tid]
        cand_valence = float(t["valence"].iloc[0]) if len(t) else 0.5
        cand_energy  = float(t["energy"].iloc[0])  if len(t) else 0.5

        rows.append({
            "track_id":           tid,
            "recent_skip_pct":    recent_skip_pct,
            "recent_avg_valence": recent_avg_valence,
            "recent_avg_energy":  recent_avg_energy,
            "valence_delta":      abs(cand_valence - recent_avg_valence),
            "energy_delta":       abs(cand_energy  - recent_avg_energy),
        })

    return pd.DataFrame(rows)

"""
Extracts contextual features for candidate tracks, such as time of day and platform.
- Time of day is bucketed into morning, afternoon, evening, night based on typical listening
    patterns.
- Platform is categorized into mobile vs desktop.

Returns a DataFrame with one row per candidate track and binary features indicating the presence of each context
"""

def _context_features(user, track_ids, time_of_day_bucket=None, platform=None):
    tod_buckets  = ["morning", "afternoon", "evening", "night"]
    plat_buckets = ["mobile", "desktop"]

    rows = []
    for tid in track_ids:
        row = {"track_id": tid}
        for b in tod_buckets:
            row[f"tod_{b}"]      = int(time_of_day_bucket == b)
        for p in plat_buckets:
            row[f"platform_{p}"] = int(platform == p)
        rows.append(row)

    return pd.DataFrame(rows)


"""
Computes a hybrid content score based on multiple similarity metrics.
- long_term_sim: cosine similarity to user's long-term audio profile
- tag_sim: cosine similarity to user's tag profile
- mood_sim: cosine similarity to active session mood centroid
- cluster_tag_sim: cosine similarity to active session mood's tag distribution
- tag_jaccard: Jaccard similarity between track's top tags and user's top tags
Each component is normalized to [0, 1] and combined with specific weights to produce a final content score.

Returns a numpy array of content scores corresponding to the input feature DataFrame.
"""
def _compute_content_score(feat_df):
    lt_norm          = _minmax(feat_df["long_term_sim"].values)
    mood_norm        = _minmax(feat_df["mood_sim"].values)
    tag_norm         = _minmax(feat_df["tag_cosine"].values)
    cluster_tag_norm = _minmax(feat_df["cluster_tag_sim"].values)
    jaccard          = feat_df["tag_jaccard"].values           # already [0, 1]

    # cluster_tag_sim and mood_sim together represent the session mood signal
    # weighted slightly lower than long-term signals since they're more volatile
    s_c = (
        0.30 * lt_norm
        + 0.25 * tag_norm
        + 0.20 * mood_norm
        + 0.15 * cluster_tag_norm
        + 0.10 * jaccard
    )
    return s_c.clip(0, 1)

"""
Computes a collaborative score based on group engagement metrics.
- group_listener_count: how many users in the group have played the track
- group_avg_ms: average listen duration across the group
- group_skip_rate: average skip rate across the group
- weighted_nbr_avg_ms: neighbor signal based on how much similar users have engaged with the track
Each component is normalized to [0, 1] and combined with specific weights to produce a final collaborative score.

Returns a numpy array of collaborative scores corresponding to the input feature DataFrame and total number of users (for normalization).
"""
def _compute_collab_score(feat_df, n_users):
    pop_norm  = _minmax(feat_df["group_listener_count"].values / max(n_users, 1))
    ms_norm   = _minmax(feat_df["group_avg_ms"].values)
    skip_norm = 1.0 - _minmax(feat_df["group_skip_rate"].values)
    ms_nbr    = _minmax(feat_df["weighted_nbr_avg_ms"].values)

    s_cf = (
        0.30 * pop_norm
        + 0.25 * ms_norm
        + 0.20 * skip_norm
        + 0.25 * ms_nbr
    )
    return s_cf.clip(0, 1)

"""
OPTIONAL: Builds training labels for candidate tracks based on the user's play history.
- Liked (1): avg_ms > liked_ms threshold AND skip_rate < 20%
- Disliked (0): skip_rate > skip_thresh
- Neutral/Unknown (-1): everything else (including tracks the user hasn't played)

Returns a numpy array of labels corresponding to the input track_ids, which can be used for training a model to learn feature weights.
"""
def _build_labels(user, track_ids, liked_ms=45_000, skip_thresh=0.60):
    u_stats = play_stats[play_stats["user"] == user].set_index("track_id")
    labels  = []
    for tid in track_ids:
        if tid in u_stats.index:
            r = u_stats.loc[tid]
            if r["avg_ms"] > liked_ms and r["skip_rate"] < 0.20:
                labels.append(1)
            elif r["skip_rate"] > skip_thresh:
                labels.append(0)
            else:
                labels.append(-1)
        else:
            labels.append(-1)
    return np.array(labels)

"""
OPTIONAL: Learns optimal weights for combining content and collaborative scores using logistic regression.
- Takes the content and collaborative scores for candidate tracks, along with the binary labels of whether the
user liked or disliked the track.
- Trains a logistic regression model to predict the labels from the scores.
- Extracts the absolute values of the learned coefficients to determine the relative importance of content vs collab features.
- Normalizes the weights to sum to 1 for interpretability.

Returns the learned weights for content and collaborative scores, which can be used to combine them into a final hybrid score.
"""
def _learn_weights(s_c, s_cf, labels):
    X      = np.column_stack([s_c, s_cf])
    scaler = StandardScaler()
    clf    = LogisticRegression(max_iter=200)
    clf.fit(scaler.fit_transform(X), labels)
    coefs     = np.abs(clf.coef_[0])
    w_c, w_cf = coefs / (coefs.sum() + 1e-9)
    return float(w_c), float(w_cf)

"""
Combines content and collaborative features into a final hybrid score for candidate tracks.
- Extracts content features, collaborative features, session features, and contextual features for the candidate tracks
- Computes separate content and collaborative scores using the defined scoring functions
- Optionally learns weights for combining the scores based on the user's play history
- Merges the scores with the original candidate information and track metadata
- Sorts the candidates by the final hybrid score and returns the top K tracks with all relevant information for analysis

Returns a DataFrame with the top K scored tracks for the user, including the hybrid score, individual feature scores,
source indicators, and track metadata.
"""
def hybrid_score(
    user,
    candidates_df,
    session_mood_centroid = None,
    session_tag_dist      = None,
    time_of_day_bucket    = None,
    platform              = None,
    use_learned_weights   = False, # If we set this to true, we can try the methods that learn weights from user history,but for now its just set to 80/20 fixed weights
    top_k                 = FINAL_TOP_K,
):
    track_ids = candidates_df["track_id"].tolist()

    content_feat = _content_features(user, track_ids, session_mood_centroid, session_tag_dist, candidates_df)
    collab_feat  = _collab_features(user, track_ids, candidates_df)
    session_feat = _session_features(user, track_ids)
    context_feat = _context_features(user, track_ids, time_of_day_bucket, platform)

    feat_df = (
        content_feat
        .merge(collab_feat,  on="track_id")
        .merge(session_feat, on="track_id")
        .merge(context_feat, on="track_id")
    )

    s_c  = _compute_content_score(feat_df)
    s_cf = _compute_collab_score(feat_df, n_users=len(USERS))

    feat_df["s_content"] = s_c
    feat_df["s_collab"]  = s_cf

    if use_learned_weights:
        labels = _build_labels(user, track_ids)
        mask   = labels != -1
        if mask.sum() >= 10:
            w_c, w_cf = _learn_weights(s_c[mask], s_cf[mask], labels[mask])
        else:
            w_c, w_cf = W_CONTENT, W_COLLAB
    else:
        w_c, w_cf = W_CONTENT, W_COLLAB

    feat_df["w_content"]    = w_c
    feat_df["w_collab"]     = w_cf
    feat_df["hybrid_score"] = w_c * s_c + w_cf * s_cf

    source_cols = candidates_df[["track_id", "pre_score", "from_content", "from_tag", "from_neighbor"]]

    result = (
        feat_df
        .merge(source_cols, on="track_id", how="left")
        .merge(
            track_table[["track_id", "track_name", "artist_name", "artist_id"]],
            on="track_id",
            how="left",
        )
        .sort_values("hybrid_score", ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )

    lead_cols = [
        "track_id", "track_name", "artist_name", "artist_id",
        "hybrid_score", "s_content", "s_collab", "w_content", "w_collab",
        "pre_score", "from_content", "from_tag", "from_neighbor",
    ]
    feat_cols = [c for c in result.columns if c not in lead_cols]
    return result[lead_cols + feat_cols]


"""
For each user, builds the final scored pool of tracks by applying the hybrid scoring function to the candidate pool.
Saves the scored pools to CSV for analysis and debugging.
Prints out the top 5 scored tracks for each user along with their feature scores and source indicators to understand why
they were recommended.
"""
scored_pools = {}
for user in USERS:
    session_centroid, session_tag_dist = _get_session_centroid(user)

    scored_pools[user] = hybrid_score(
        user                  = user,
        candidates_df         = candidate_pools[user],
        session_mood_centroid = session_centroid,
        session_tag_dist      = session_tag_dist,
        time_of_day_bucket    = None,
        platform              = None,
        use_learned_weights   = False,
        top_k                 = FINAL_TOP_K,
    )

    scored_pools[user].to_csv(DATA_PROC / f"hybrid_scored_{user}.csv", index=False) 

    print(f"{user}  w_c={scored_pools[user]['w_content'].iloc[0]:.2f}  "
          f"w_cf={scored_pools[user]['w_collab'].iloc[0]:.2f}  "
          f"pool={len(scored_pools[user])}")
    print(
        scored_pools[user][
            ["track_name", "artist_name", "hybrid_score", "pre_score",
             "from_content", "from_tag", "from_neighbor"]
        ].head(5).to_string(index=False)
    )
    print()

print("Hybrid scoring complete.")

andy  w_c=0.80  w_cf=0.20  pool=150
                       track_name       artist_name  hybrid_score  pre_score  from_content  from_tag  from_neighbor
         Running Through The Rain           YUGYEOM      0.784124   0.708342          True      True          False
                        SUMMERIDE          Jay Park      0.770534   0.699218          True     False          False
                  Didn't Cha Know       Erykah Badu      0.766561   0.871330         False      True          False
Frontin' (feat. JAY-Z) - Club Mix Pharrell Williams      0.758643   0.895538         False      True          False
                       Can't Wait          Doja Cat      0.753219   0.729839          True     False          False

dishita  w_c=0.80  w_cf=0.20  pool=150
              track_name artist_name  hybrid_score  pre_score  from_content  from_tag  from_neighbor
        act ii: date @ 8       4batz      0.740810   0.739676          True     False          False
                  Jammin  

In [ ]:
# RE-RANKING

# UI-WIDGETS (?)

# TESTING

# DEMO (?)